# 🎬 Flock Clip Engine — runs in your browser

Turns one long video into captioned vertical clips. **Nothing gets installed on your computer** — this all runs on Google's machines.

**How to use this page (once, top to bottom):**
1. *(Optional but faster)* Menu bar → **Runtime → Change runtime type → T4 GPU → Save**
2. Click the **▶ play button** on the left edge of each gray box below, **in order**. Wait for each to finish (the spinner stops) before the next.
3. Box 3 is where you paste your video link and API key.

At the end, a zip of your clips downloads to your computer.

### 1️⃣ Install the tools (2–3 min). Click ▶ and wait.

In [ ]:
!pip -q install anthropic yt-dlp pyyaml faster-whisper
!pip -q install mediapipe opencv-python-headless || echo "(no face tracking — will center-crop)"
print("\n✅ Install done. Go to step 2.")

### 2️⃣ Load the engine. Click ▶ (instant).

In [ ]:
import base64, io, tarfile
BLOB_PARTS = [
    "H4sIAHBvTGoC/+19+XbbRtZn/uZTVMOnP5MxCXHTYibMN7ItLxMvOpITdx+Gg4BgUUQEAmwAFMXoU5/+ax5gzjzB/DEP1k8y93er",
    "CgtJeelJfLrbRHcsolB1a79197L37L3/dupeP5fuWMZf/S5PUz13/W02O938N9JbzXar/ZW4/uozPIskdWOq/qsv82kfiVnqz2S/",
    "dXjUaT982H24bzdbrcPWQavy1e75t3+8wJ87MrzwQ7nnOH7op45jz1e/+f4/6Hbv3P/7ze5Xrf12u71/eHDQPaT93223m1+J5ufc",
    "/3EUpe/L96Hv/6KPZVmPaQmIE14C4u9/+98ikcGkMY2SVI7Fm/kisedx1EjSVSDFlYxT33ODBpaNiGVIZ4aM7UrlPHUvZNITfkh/",
    "UtH4TqSxGyZe7I8k3q4izx3hx8QPx84smskwTfDuLTh3LCexO+OsnjtP/ShMVDJqsCsnrjcVs2i8oDb4CdUylnN8CtNgJeJFGLoj",
    "+uKGY7GM/VQiRyrjmRz7bkrp1OiJ61GFaSSWUXy59+0v0ei7PZFEVGZVSdB4qjcU1Fg/TObSQ9+jmOrH6qAXu0IDVak4Do1AQq1z",
    "HNEXVtNu2U3rXxpP2v8U539n8/xv7c7/z3L+HxbO/06zedTu2kf7zYdHh7vj/0s7/xepH/zWZ/9HnP9dIgDU+X/QbHU6Ldr/nS72",
    "/+78/yzn//nUjem4m8pgTodbT3hROPEvRBC5YzrO62Iymc3lxd5kQnQAHZDL2J0jY10kweIisXEwTuJoJhxnskgXsaSz0Z/Nozil",
    "wzWMUpdP80pFp/2SRKH5HUvzK5li8WVvixHV5ckkUZDnbjoN/JEBe0qvGbyVOwsqlcdvXj998cw5PX77nA5mZKhSc/yAGlOz6RSP",
    "gitZrdlz6mmY6j9iT1iqrxZ+johgGdsARwd9ZSwnPAKOylFFE3oMWPyXSNKYainUWQOxMva9tFcR9Cx9yhYRhcLFasJNxER9wRNL",
    "GqWQG24n7kQ6qKc6qelaMar+ZFVN5XXaQ1V1MXOvAxmCukqp3oMmV0dfFMyE0mJp06BVY2vwP35a/pQ0hlZdWPQfgNhBtJRxtVaz",
    "qYg/r9a2lPopcRrDByjUoH8SlUU3NBn0VAOGGgDlqYE6soA8zGARFVb1ZuOeCPwkHVDGYV18/fXlUrU1m1D7cTSbB5JIqlOVoPpA",
    "i+hsEQq3kLMuYtdPaAWq4QRZuMA6TVIiVGIRhWLi+gEl8QoEEBSkXhUq042qm9JOtEjni7T/Nl5INTj6J7eUgfgThmOrznvRWIo/",
    "9EWzMH3UKimotTg2T+I4iqvZNzwTWlSzGWhRtI8aXL1ZA3hb64mb++K+/Uvkc/tqtz+FKpPq3KDRJrzYG95aGeTSjCCrHnbek86Y",
    "mhKCLk2qV/5YRmql8tCnCxruAa2cOpbPMB9uBaq69MfptE6737+YpjStE5FOJZHpMZHxDAsLTbozIvJR8pQoddr8QmODb6iTQZCI",
    "ketdgrqmncVT9rNCGqLh/2wmbDmVoSlGRHx4PwWtnVJxOWbQ1YTYAo1txGjhB+OE8ALxGZh/mjjhpzXbNN/MlUIc9nLqe9OqpcFb",
    "tXy69KLAShiU5inLjEV/hX8lJpNfiQUiHsBRPU+QdNVrWvVy+UYyjZZ0dtKekJxHZe+rEVUDytCiCf4A8WFvpbGao1oObljLfioY",
    "1GBkt4EZkqpZGTQGtYFlGjUcNIfrOIVmuKq+DyxuhjWs1UupullUI5e9J57SBGD2embk57EP3kw3xA8nESZWTyKWtbz26XsYhY1f",
    "ZRyJahgJta0EEK6eou17cWCpSnhY/NJoDD+0SdUYzTTakm5MEx5bP/Jyt7/+z+pP45t2ff+2dp39IviFTZVtbzqTxOy9+xlbeBGM",
    "OedYgpMEb5xvMuySG272rVXamBjpmX0RR4t5tVXTQ28S2jWD4plJdTDDhXOlLqLRL7xlX0ehVO3DV1vlxjhUeVGMF7N5UqXMdWaE",
    "w7TfrgtqGYbOTTzf79OcJjKrjeZxvF6Z3iC62YW1xkcnn1o2l+NqCdSOUv73fHb8/47/L8r/95tH9uHDh4etHf//xfH/Snj720sA",
    "PsD/tw4OD3L5/8E++P9D+rPj/z8T/8/i7xbL/l+oJSCeSeLeiQH23EAk0SL2pD2bdwVz4674c7R4uyAq/oezl+ADTUaQf58qDtCM",
    "/91sPn+xkcmkEyGpKRu1YKuqgZpXhny/wP7gxzrPg3pA05Y7qLgdWzyJliHTQqAVf1bffobWwUV/M2YT9dizy7EfV5U4IdEEK5HH",
    "SepElwWiFcxLn0tA0JCPp1XJ2BiVxtqGBPxOtWpN03Te29sDrYyfCX7XCpzNPfEIupZXp12xmKM/reZRc87TSCxfmK7EhCbHqGyE",
    "F0dzYqovpZwnPFVE47uhtHNCeJNFWqWNcTDfYHyYnxlR5TxkA8VTfNtH/cMB6HXq2/ABMriLsR+ppK473ENSloHfNmA3ZjK+kA3F",
    "AzSoBzOXuSgM13reSHMQYIzK39R4bnBYkujifACT2DPCIpW/ZsvrObE4iwTykiyf5hkou82Tm1QLs5CzEE9pTF9H6dNoEY4VH0El",
    "SlAAIRNGQaxADc8TyjA1Y+tF8xXg1JG3xGvQ+29wRNs7+4+d/Ueu/2m3ugd2s3v0sNs+2BGAXxj9l6vsf1sa8P30X6vTPTg0+p9u",
    "s9mG/ccB/dnRf5+T/msz4fA2t9pgqfu7N2dPGi9Pfjx5KYAlaKxm88SuVN5F8biQIogCUnJjHH5M4hlJ8nIaBVKoFdYj4sj1Ujb6",
    "mEcQMxI94sZudCkzs496BTLGZC7dSxk33CUgK+sQyJXdIBDK9gMKgMswWrKCAKJlCRuRJdrlg7Lxx9TMR653SXlZydAgGiK4ttTp",
    "+vf/+b/Eq5d/Eu+mPlVFZOOz0x8arufJQMYuG3+E4nhOZJQ49wPfo7fqK5m6Qc1WRFcEXcUr17MV4KUCQ9ABuGBAUxehlERKPv7h",
    "yTGTY5RNjr8BrZUAyOPTH5gooR5QP6t+6AU24NY04DFRaxfUdwUYjZzEUhIVOXdXDaqxQaOySKU4Pn0hqgkEtoGcpFzTKlqAKCRK",
    "j1pdqbxRItrEm8qZK6oYqMSG4E/THTcWJ1k9MVA/6Zc1cWkRTBZBKBMt4Sb6lD602nanC4l5OFZvD4/wVU0aCp6fnhx/f3LmNJvW",
    "bV3Ytj28rXwiZxAln8QV5DJVzRzk2KyoEykyCHVWW1AfmHUQ/8WCV6II8adMTkLwHI1l0NOqPytwQaRetWlQxr4b+7/S2h5FUUDf",
    "mOwvsx73xBM5cRdB2uNFd8fawvTW6rwczLr8k6LOdTMJuPlF8xsl9oVMZXhVtR6/fHHqnLx+9uL1iXN8foaZchdpZGUi7wxAX3/J",
    "ac0cttoelNvxE8dFA51ENZCIVW5Vvs4rW/iafElZ2+oF9LxazkyFHUpW81NXQ9znfw25XoDh0zjRgqRdxToa9cvRLSpqfDLQ5Swf",
    "VwtamvVyC0zzbQu0bCn09d87wGd7egt4800rRbYwLYrX+NENFpmyYhECEYZFk7tsYd/oX0ZRUVA90OzVC9ue/95usBhqM20dS7OL",
    "8q3B654VwFCG58pGVbqhS/NJE4BnJPRIbDj9DGEQuKfGTR0fhOWTusLnhXMGakHoDbU28lkUjSkn7y1Geo+jwB3tvfPDcbRM9l4S",
    "crzO1IeuIEwWZFvLaB9t2obhBe1Gjb7KKkZGNOXOG4yjAb1C71VzxvLK94BBLG8xdtVemrqJg7dsD3nzhQLtRbM5cDdlnwSRm7YO",
    "uIQBkkFRxei0PFLlElpaVOgmR0K9EkIyvzuF9A6z0HLsL2ZI1b9ugUGqevGaRK0ZhKKt2MMqqq3r1vXVn7rpg5Ou5rKvX7R9gbxg",
    "I8+6gKnkzC6g44IKlCfYySdYy1Cu3DGsN2jQtRSlku2SXmF9EeCB0oHy0SovgCRMxfmWwccl4w/6Zqu9RkmD4RrLrRAYrFvCcfVm",
    "/QAQ5lRcMghjTFHfzGdOyRjEUJXntrpUsh3qcWdbEXWQlgtQ2l3Zt5+05Yzlrcx9M5s5w7kfv4MLxBJv3w/SS1c+EUg41vRWfR2J",
    "4vauJqVNJ9wkWdACrH3DszWj/eyDviE8RZTi2KMNmAhFJIGuAVGlVpkuTruetkldLBKpqU6FpjPEy9Ts86fO2zffn7zmrS/d8Zo1",
    "gdrWNDhmpxsd+St3TojAJzjBSo2VCF1asqCvMDAw+FiEfrqi0Z5HyTfC7DqQomjOXxaE5OizRlW2npl5hH2czVppQ1MzGhncPd2g",
    "hsnRwGla3yjZ+WDJznrJAlLYWk59L5RaQxofrkzv3lgmwNL94gAXsULRBCJDDyD+nCh2phMHw9XHP3cgjUpuJvNxiEI1iHtjGaRB",
    "aLKEFjLUAczBOdVxWd/EHoS6lxoY7/8aJj9X5Rcf2h8pHU3yH8I+A/Vr+KkIaKCTh5+AgwacOPyd0NAWMpOQD8joXnFDzgM3hRy4",
    "ZAGl0+xkRTt5RiVxXj5xY+IHLbZSyXLMiC8k3lNncePZQdfYrK1Rc5vYcI2634ocDf9hMA7hC5VEWDFwf/UJZSQRY4GYOUfFFiuG",
    "WPGBhkBJo5hY2Nwo6jPRFFzQTcDXmE6wMYjD42B2+odPfJb1bwDh1MKRXysjA6q3iAQ4N/iy1Js6oDj6rYPcVInGhwY1oxeVhwgx",
    "TlhVjUBeEV7OsYJqE/I6ugszOo4225fnqAZueLFwL6QDK72+auPAMqnWcI38WetKBpdBVk3xDLVQ8Y3m1IXusR7ejT1WUjPw2ne8",
    "qRs72SAk2tqnUjwPwUvc3BouTK9h3hYFrtGchkXeibIWe/IkP7FP/TlxMrSP6IilSSW0nEZ0sva3Atw6UBp+RgL08aZmvFYwZNsY",
    "zSTBoDHON92rFuDUdZlPIhI3J+cfxfkaq7KGiMose1ipi5DnhxYrD1IiqrEby5pyr7r051/AiWDOQp1QLx0RtY8+I8oMcUGxu4VG",
    "fR01iCylTUZ8Q0hr40raUMoFPC+g/4B4n2iA4vzJ97BdvHIb7QzNbzCcNU29voRUzU1oCwl5PafTyk8Fi9s0avemERBslIudWJqZ",
    "kZ8hISeiUmks2DeNCxnskZ0Jc73FeJmgwCqdQrxJvGyozTGjEA5vEtrpXNxSJmOVgOB1lL6AoTVWtxyvmSlb2SBk7aPVu/RhXr2S",
    "qS1+SDIRQv++2Yn36+CafANVgBqnkz+HaubKxtzI0J7HlMOjZkpNfyczWm6O0uk+0EP+YJHSdLmhJxNGUDOitc0kUOcT29KEnV4Q",
    "hbOvRCyk8Srfm/pM5gN13UaWE22AsEGAXLl+AO9Bre+V156cp+KE/9AMbpjtM7b991GN7fS/O/1vyf6vc2Q3u4eHraOjnf73C9P/",
    "spP27+AA+AH7v87hfu7/f9iG/rezv3+40/9+Vv1vh6mOp1D47c3jaLzwmIDhNbEI3HhFNGpMxzlSoQDWssCZCzFeouVh7IEfxey1",
    "A7cJInyt48SdT0H/PQ0i71K8dYNLeoNPPlE+XKxmi3dwOboGFQSXkwqL8EAEaLLn0cnTN2cneWiAkXK24CYjrxYmjvwLDj9gxG1L",
    "yhKB8EIYg7pgJyFBBEQljASRky6luCkAJFKrYIhKt8WTmEhHIvSIJI8l0VujlfaJ3Mt9BLOgBg61ewGd+CeqN2P5aUaPxomjvqnp",
    "hBBl5XBrlF6XPxpphmp6j90T1/SSY5cZ4txBJC+tqCHVN8qigCiSvthtmsmbW3YGzNlN0JLq6wbxlMM3PP05OkcDPJ/GLs1CQKiI",
    "CMWxTDyQ1koOzCuBIEospwmvohSrqJZPsF4AiiZWuskYsSmg10xYFlPV80RjN0uIRxKXctUP3Nlo7IrLq55oUNXVy6tBk9goIvDh",
    "9FIQJoJZpcEaaFZQ8ZPwjXE0g0dMZsaH5Rwk5x5WMu5zGUfwqI2h+8J33ch8nMAVoyGcMWtIyRIRUpxWmQ0t17cpb4TMKCSqu5o1",
    "kYVhOg1V1XpbJQ95lyg3kfOJjK+k49FM5aB0d/JGlvWVWoKeTWRPJAHtPeFi8oizYZwRS17Uai/P3VCZqYL3gTS5xA6U+W9qV7PM",
    "M099xAgR3+pxxIDQqhfhZgd1/X1hCUu5YqoBoV2CAeUJKkzxwO/54oEIC856RdGvBvYxoxrLeeB6ionqq9EzU701P1rzC8QcczRJ",
    "hosZK12qBTiDXji8o7ZsDQ/Q/F+GhSklAFuLUMa+CDc+bU5sqUCr9OnTNkdBLZ3jiDrvuU3ZQBH3qdFOct1V5hatS5Q8o41bNB05",
    "2inagCkv7ogmxQ9dIwQuDPSWimj2TX7iKBdwji8a82btyGDYOs966UGvNfw4CJxT56NJLX5p9YaV7YUqO/5vx/+93/63ddTtduxu",
    "p9VtHrV3/N8Xxv8VY3P9lmzgB+x/m4ddY//b7Rzsd2j/77e6O/+vz8v/dRX/58P4Fvar0Ei4PlQYHOvNhGxjOe7jwF2MJXGBb6eI",
    "xqZsHKwrP1ZsV0KcorSUBBsWlZBiX3K+RHp0LonEXXjEZ71I77NE/eXLV8yDgN6DXDwXoWviECcZPjLgcX29gUaJwz0YQZG6QDIs",
    "UBTfMrgPLZMMPXl/WFEx4MCiNuZukqoob1EICbSxC4FA/kKGMvY9YXpl2EfqBRsxE1+o7FGoElpD0ayiVboj6VJTOHBepaI8zhIe",
    "Qz14J386fvxWsOZlT/JoUyPHCKcTQtUDlVGu1RQjNqhGhAnwQ94CkRcwsBWeFVckq4DF15/Mfxbj8Hyiqe0dvOirN09OXhKtd4dh",
    "Kn8G/eXx6mlQmVCmjX3L0GBF9LOFjy3a7RZ5Wg6P4/AAZxFy1vhc5bYYptM4mtOU6m4cm4RKic/byg2XGT/N5K5xW1sCSVhhpOHC",
    "3ZBIvNWvSrsz9sdrmqJ4Ef6nMZUpbADiJzJtlBw7+RfN2iitP5C4N7nIGPUBh+YxzVU2EvSxoAH6c7Rgo31aQVMaDHa0Uzud6Fpi",
    "Li58uIEi12QLSjDYwA/ZiZK2ZkENpHQ+ajO7GCWE0mAjYRdhRWhLuwHQQu6ZiLaqyjTVev727AVtkv9+/uY1q7wyJRD+hXcedWZC",
    "C/5Yb+ueuNnc6beVCqMz5Rx5k62SW70bwd4R3ioM9kgSV2ALjjbJY/Hqh/O3vUoDcSFHMl1KGVJFerAH94khdfTupdoYVxW/Un2F",
    "r/onjxZBRHQo5UUwjaJLNVKBiwxAEn6pHuRwZoRkHORwVIYtsAmsniLx5vUJW2og0pIWdQH3uAqzENrB9FSxiKn7foC3aDKpEQSe",
    "IMEz1MPqwPBc0IBzeQ74gjqIuRMLxObk7DSWKQ02oVKZDZ2aSeyOTPlshkDpknNjvWyBl6MPYYGzoljrodeL58WOn749OeOSgVsq",
    "mPppIC2ghQRc9l6qArjUefumjAyq3x40BUwqElUEg00l2EaHnVUwUxgfaGY5x3K6ogxYwQmUqzg3aCIplUZBnyUIQcoD9vb5i3Nh",
    "liSXVodjT7Qaraao8uRDyMlirwfZnMHhgl5NyfyIUo289HlELKXlttidg05YLwrGKjIXsX2LeB4l+MYRmyQrWFm+Mo2WNXMy8Raj",
    "3XOjEUZPDOCtIYa3t1puBuEct5mFcjbsOOcxAcbpf3b8+vzx2YvTt6LKEx+wj8+A57tBkzZkKQBxsjf5JrvlQKoaaykJSIaINUt8",
    "TzzD4RstqEeSF2dP2WTpQSWYCP46VoQFEwbsEaSlDSM5ibQ7EiMQqnlpzLppHOfAkly1TesnQdxam6gSCFQqJWePPp9Y6nhRoPut",
    "AyJY6xqh9tWf3JTBgOsPbqw44oVnAVnxoYdNE2IXIOV2WDRXvCdeqd7N3JWQMz/NOzaCtLPUJd56nMwHCeRWyiqAP+hqMgPTYMUR",
    "fCDk0jIuOprdNI2rDIIahs8sFVFSLwVZWcjMbQ0uP2XYSQFBvrSwJq+iVj5zTOaBA7KGxo8NeJKqp0wsElWbh5o461opb+0zjlxv",
    "uF24dU+MafUQOmowBQVqKAdmYxVXC7Jeryc8bZrCO7EumjVIeBDYVxZ89k1T+C9HwHN0Q7e4vqiqcteXTS8Ls73UCXSXl8X7jvqi",
    "TVNdcEgpQ++02mWJFPYhEZQ+mz7VtYQ0k4wWBKM58eJNF+Gl6tHYiDq5jmHBLpe2NSJNQajO+QfNYWZDVNdJjdZQ2whViqLAoph1",
    "iyyQy+bST+6AsXyaWIMbrqRntye3jRv4teDXUNwAsvFsyeSQ3OziAFs/hbpmhpvJ/NZWJs2MoSg3BjwL8Ki9xDKbo3OCQfX8ZSE5",
    "XnZO1isbHyLWXTbxoZNksUnTZ9EkStYrDASG0r6ep5KyYknH8igpWG+JhlDGWehBbtNVq2W/c9E8z92H4PLsrUFVVl+Amc+ttpWp",
    "fi9XTO7WC25Jm+JLDJuhntGOhu7nt6Jzd95Sl+qi0BZQ6myYppdlB7uYE+gzXjVqJWSHoynTKGJOlXwfvKuieKNMs2jPV2IcSY5O",
    "GBOPwmHmCi3RXABLtNck2RzggXvUEC3Z6EAFUZikb7nTD/hTSUILwJkUOsesWfjPtdiitGJOrgk3eGl+wkWjXyS96+gsykGBMTMR",
    "SRF7aOAsmcjMyCpZxDxWrGHh0zxbiGqvctzQYrhQ6pxOyyKUWD///HPRdrRUlNUZKsegNbRjOYuu5JxIEf+6qkIhlsORljbAZlg6",
    "PmGKi46/ofNPJIx1ec31tqMqbhC4mKp1A9NUfo9VQgl5mPn7Q180WirUIf33nUpdi/ixtYWK8unpiS4oij4cM7QwaSLns4h95Amu",
    "ar3oDVA3j8XtN5pK7jQ1+dpT6HDQo5ThH+LbWjF46M7+ayf//wj5f7N5dPTQbrcPHh7t7+/k/1+Y/N9bpL9H+PcPxn/fP+hk8d+7",
    "Kv57u7uL//555f/7zE4+piUgzgPYsrNZO8gjsDpaMKPDaomIqSU3YGN0OtlBR/CVK5QmxpCguz5fC8PW4kJfmqKEJjrMMJ39MGfX",
    "0gI+q4d2BZycYyDBBZioF6KCQE1kDbqfGIILhvQIqEH0HRyp6ooRvHDnSswmQeS5YeXnC9D5U6LDp1Ew/rnORzsdsp4y/hJzX4Iw",
    "UsZkSqGhRPszf6xuurEWs1li7c1dYt4TTctT36/c0E+mHy96/8ei3NG2zELcafF3mVW5Qyxe6nVPUfOwl7EP18Tja7wsLQGOTbfJ",
    "891JjGdx7NbEz4rR3japVk3NAuUr+tgUyMlMq9MXjppkB5NcLRHia73cGuUthzOo5v2p5Y0GUWVy1dbMq2hRwZvOfC7F3N4WUjyL",
    "br1SgcRhITexbpKe3ZncclIaqSSZJ5lI2DoK3XqYO6/H4clp0Vy3D7pcwuu5HOXD9fiVrVdYjAOdG8IVbI2MV6RIC7KHoomW7mhP",
    "7U+W6lE/69gnod4ztvieYwge7/3IXlKr0IOZltpGtoZ2JhtSRc+PTDklb254AjHtauC6sqlhsV7hVifx7M1pomRbXiBdaAB0POxY",
    "TWTu0OXXRRWTVCvbRmXzWYgEny/yiQVIzo3fa7bHt/ly/z2nNQ/091tO8Hwr2MJE85AZUcpcy7Z4Pvh+jOLGV7ObXqeWnsTjEYGG",
    "OylwVsJS9LE6GJiLNr5PanbHcra4lkqbSu1J6MeFG48DSLLp+PCW2t2+UHkxtLmRT04sbtb9m0LPbu//FFo822wBx10yjqSl2dqc",
    "qYkWvnoqimQDF27gR7M0Q8U2bRvP/989uAF0eJcEsIjpton93LnG5LkTnLrhQStF+M8w94l7HAUBnYeyeGRqZ0PI10un5yiWLkud",
    "acsmNBQsLvgOdRrdSyYmQAHdMv5HV1vemttMUv2J8nlmS/GiCAupLDxsDSE5QUfLHHfhOwtWyjLGTdtEFDDrfrAsnFum7NCIiJIZ",
    "worN3THLQ2hVL6UYRxAB8eGPYcHyd/lQFnJ8IZPi7A2qM/e6CrUAdaNpN48QxEk8UD+V0ykSfNbxJsN/LrZqx//v+P+c/3/YbXW6",
    "9sPuUbu94/+/OP5fX8L5m8sAPmT/12rtM//faR4c7rf5/tfm4S7+5+fl/w+Y/z3T97C2DnoPQWA87LUOtO+2jrQE1QMTCVXQflOi",
    "8JgeY4M3mAkF7DN/PJ/HETEPPb4/R+sqJq4nFUuhKqm+wgWtCCyBSD8+3HqmOHlnfugSD4DsWbwR4YoUQYXSCuKYN66FB9uLWEWe",
    "8onwSWY0QUrPgxwqyDv7BgG9iaq8nkchXLoQ8J3zUieYC/FTcQGXlKRSMP/7xYdbOt9/x6xPIFMjuWB3KyU4kHGDoaNG6rW5Swm6",
    "V+3cA5qkxVYTfBstXPzzCC/izRlE/OhookKnKtKFdblu6nu6l6pHVTdYuisWflyy4VfbFk/Xi77/AWDuuxxzrQ2eTdh4UgWVymnu",
    "97eAJ/9YUf49kSyJBmTZiL6EF1TNsyiCz1U2ieLrr48XafSUMMrXXxtzSSKqkkriwt7CWxFRRGx62tAK0JXg8K6wdbEFm5Iy6a/t",
    "SRFe4RsY6qzHQPCmsEhKfh+xy/p9anUliHnzw1vnXV3gDy4YRKT7umg9bDfvHut7uFE4CtnCLTd1c8MrNxFEMabelGbO6B3zu5p4",
    "a7CQw7kzbOl7XPrKshyD0TMGl/qA9m/cGpfXp9VxvNcceEfhCiulWH0uvhYPCWzroFYT/yH+2ir2FSuUrxwT1Qni9MD8N+R1onmm",
    "WiZvyYB/1xfveu9fsfeEG8Aic5XfPe0ntCK5Pt6ZcACi9fWrLHPwm6xgxuzlfV2LYGI1ribMxQN6/wZ6chow09zabe/m+W09oTbI",
    "/g2vCErhJXFrbUDS3CFkHQVZzHskMGqrs7hL70pHJxVaXOcZNCugdKWZzly8H+J8E4lMNIqq0w7nnmSWCvlqVYs0g3MNy9F3xNtk",
    "AyH29kT7Nx9uA56HuXdzfdtr/m6DfU88wgWDOFVoEzSuCBkBy/AQXTforCCGHTtD6dQznGnmCKbl+jRIiJf0ZmMj99KvOuwz7C0D",
    "ttFj1BbyqYfb8/h0Ym1/jNgxl5KjQd/v3Rcy9eg0VEwo2E0cvhq2txz3lR0tW8Vrqz6evJGbMPS6MfMJ6dDc09fy0Vmygm0Hy3K0",
    "EGbGts6Z6AfnF6UpJDF1gwn05mabmslm65kyi08Hr3fNBjzrqw+rRvPFvI8K64fLNLia2p02PzfK3EdPibi57tmtye03Wl+P9pek",
    "R2smPgXh0MeszO2yuWx96kntT/o3XDFG+ra+bdU2P2bN3rFeC42AdJ0mGx8wRbU77cXWEEXpxHjH5mGELvTfsnZgMk+cxIV6JlcN",
    "dO1mLbds4o+KVKNjkM3AS7SZnnOixSD9VbBw/U1GZyHyXU6V8V2GavcdhytzeatQgc9nfpIoI9tRUi+QSohrrs97dGAsPSIPkgXN",
    "sB6NhFajFkSqhQHCKl67ldTdRk4ViD0vdpOp9j7JojCVoyptCy/kXbXXk/KW0yk/y71rFRUcwW59Nrd5J7K5EcbRUR8RVwHk3BPz",
    "piLvOeouUnrvt94fHu6DD+zVs7rUxcZjYJB+097fHv0IvZDFu1RBDEw4hn6Vl0DGDSzCLJISMxJbRronbuRtzdpAygOtiSFaCEqm",
    "q7bNl3o+VreBboQOTGLPmSjTTFfpjlDk8fGpc3r25tR5enrOJtCdplZgybnGRC19E6ouv1fYAga04QvgXqCM37Q+SxNSRqOVZYT5",
    "qH20XzMxTt1ljh/98XXmnm5OEMdD2rsMpSpsDavTfJQjOpwVg6S6CPJn8zai6LIs7mThbTETqv+j7n/pAmNeziDsKMOeGc416yYO",
    "eKDXrK1vcOVx9q7Sx1GAsOMxHzc89m9evjlzHj07a589e1SrrUfII2h2tuy2RCe4l6OVvoqcm6RiFF1vZMR9UXouyzDfuy8KJo60",
    "BMc27vziDQDnctucz85I28PRj2ubCdlNZ3z6hBj8uCHtI6BsBvljSgrwr6m/xAxlVWE91Ii6frdRprxw8gX6denLA1FtwaAy52u/",
    "ptrW7qhamsO1mrHLXLoYX4KWRObJr9ZeIOEQX6sUsZjtBVGWaBzN3eVOYPiv9+zuf93d/1qM/3bQatqdTuuw3dnt5i/O/i83gv+c",
    "9n/7h/tNE//tsNnh+7/ahwc7+f9nlf8fMtmOe70ao5UKubR+NZcOgHt8fi6SxYh9K9Wdr5XKO5nJ6abulSyFbdKhdI1Qgx3cAIPV",
    "Cj/9dJnVo0I8qejtunzFA7Eppv7FlC+MScR8wZJzlsvoqGTgGt0kEdVcLFPjEHHwS4VwYryAJJ8oFXiswnFRxZIS8KmFb+/ZSePR",
    "8fnJk0wUxvZ1efuVp7QxhtF+w8YWkgMOoNBC2QtUlIsEzAWaHNgOY5X5a2gXa7ZCQ0dBqBvDSWUZJ5Y0zDCZ41gA02z44R1YGcf+",
    "JEUELRoLYq8plb2vlbMaJhB3csxTZsP49tfWNwzjTM6iDbm+st2UPEom2N1vIVVn2QRu6hk7brLhWPUxUmzNChpjQr3+tLnHmrmi",
    "QVpUlVVy5F+3LeT43bm/vPGyKlk1KjZuysQQJLEE1FFvVapIl14we+Swn5mDN+NNRVm0ZaJDCxifLF3jFtEZfWXBGZXf4voFyZnd",
    "rONz0cuOiPy88bV1vy4qpADA+KStCuf+eHeU1W47jtodaHTVwx081Jn61gIlQd2aG8sT3w2iC+JmRbN+46SJak/tVr3A7PK2/sr1",
    "w3q9yf+raxe+gquKnuKidE/Px4MNTz6ddJd4bH2GyrZUeopYOpZbU+U5hqU4YoOiZ6QuqhwYfTbtgYwKws7cuTKrAaamqkHlMS43",
    "x3OzXZKPetHYK3PvpB36JpS8gGCZeiVj7VmYfFO4BZFWfsLolbCfCrjPjv05LhUKtxIGuk88Msrcz+y7Cgg3l7NYWarDRYtbartl",
    "5rJslImxKNw1sKCRSMrCGaVoKjohFqLUE1PbajZrtZqShgNlmZgAqI/6WrpQTZmJ6dD4JlgabJRNdzgN8d4sfY9Glr2gQ3ku6fBI",
    "IDzUg5SPTc9YmWs3aY15CDbjXKDiUI+yXZTM8BLJIzF+1xcHPDf5oJeFJGqe+oXvA1/8kcHkSbVhqUzJ7HNi3dzQ6NyoEb/96Sfv",
    "hmHe3t7eoBW3+ExphL7uz2N/5sYrNcP3h5RFWO+L7fi+igx4Ud6hxtRT2XFmboB6j5RRbuF4yNb/KAowuY1WaTaXfAu5cmU30f3U",
    "pYGjwPUu+cpyKoi/D3EFgZryZn5NCN+ppuSWVk/sc/40jXA3UhuO8tGcfh2p64+yWudR4mNzsThflaWNv1/qLyKVDM5ViJEX4SQa",
    "VtTL29WccMBV1242H1ROA3d1JpM/9Vi5bF7/3GMtc+Vd7M7P0xWE9e1KZfBj94Hg12SIsBszl9bia5bHPaWlGGa/1J1tp2pKIbdb",
    "xHXxZpECe5pX3JGa/aYByjLUxfnUHUfLujg2l43UxSs3Jl7hpflxZn78WBcnsPcmuqqiG8p4Xi2qCbWF1lL9BnucU9Cy+7ylu7f1",
    "7Suv/h/PjUiAfh6ZnzeYRVMmUk1l2AouN5nfeVJv6wdN/L/Vpv9o5E6uYBWej9pLd4XrX88VRXYC12Fufl2P5/v6O5lIYOu3dEQx",
    "1WSUMnTqKbS0FXtPtU7doK69PdE5ILSW3bqHj9nXP6qPyHWg8ySM9c3ng2Z5qd1Mb3s3M7Zr790kveY+tGhWZWf/uZP/fND+E/Kf",
    "Tsc+bHcO293uTgD0xdl/Mn/8m7uAfkD+c5D5f0L+c8Dx/w+7nZ3857PKf45YfPAmhlkaIsARzSoecXwKczG7mPnXmecnZCdEE48Q",
    "jpElOhNEMBavTrvEHuFmM9aLVSrHgfYSIvJJX9GWXXNEJ3G8EtWftX7x55oKg6b8Q4IAzPCFVLQ9R9tigYtgXTDo5QpziDAahBem",
    "DlTJkfbe/PB2T1sUsi+nAorYHjAP+PRY/R8RHzGzCSjIysxvXPi2nm3B10nRny0fi4EQkUu/r2fzQ9aUuon+tSWLVlsjj/65JVPh",
    "2mjKl79tycqh/5FL3QGQZyiaUPK1ekp2U1fOg0mwuPAnqy1XFxjVsvFv5XDbRNQ5Yz8uWj462H4m4Wt1RWN2GTtfBlDWAW+GhKyr",
    "FatgvO+ud9VyRxu+rt8LnwsKAEeLCFQRaK/zrlcLcGq5ayy/K9ajmDkTe6Dj9uyS/gV7xPf7qSuR5TVV6kSXhRBVv0QjLQjj4YEy",
    "Xw208TjUUVj2rBrEQBwYjspYWemPqaiSG35Yrb0jvdb+/rf/a9W+yaRWscd0q1mFtvpZNSJSqqpWhNMmOPk6y2EZyV15DZaujo69",
    "TXAdAqdWprrjwQDMFqm9cUNGbkFpTCixotQ99Vgm1npYz/fep6Yu0kW5PXP7CYPzYQF41aZVtBgxeiU8qQwDESS2Ac9ZoK1FmioH",
    "+swgyaqs36onvWkDRn4m9iYjBSql7mprNLg+bARGl2G0tIsyteJodWm02JhLYxgzWrxdeG/C+kWjHHszKCuPvxm/QvDVtSBxbLWS",
    "w1wLD1e0JFLW6xzaRgHSATpNNFQ+WYzsjnfdhpyJpeplURNDKsQ3o4VOxbBl4P7LZI92/7W+4a96M2zZZ8UG79PoGUm/uPFvRZUD",
    "dQ7uszrk/nDQ6zaHtzUzqMogSKN5mz35sYJRpM61lk15izUdUE0Gf5vaytuufEkEMo45fm2G6G1jTk7N+FB1h+iYObw+XB/0Lf3S",
    "CWeXJf4fqu+IuweCs1xbfg0nT7uRLDnQ5yijI6I4qBIziBpnFhYk9mHZYlKBMvw5QypZShK4baEV1s6hkv1kYYNunitr2gxgZfDs",
    "GjkrhYMKTTqsFVQauj5eoDfIXfBMv5pwyFujd0v6N9RoZYhqwmnOMPvvMXddt3TNYhmrDuRy2VFB98LXs1rDgUJvDtF7znhUEJCi",
    "1geotqG8M5wgiuZcbatUORe/0wVeWcVOfNzc6agYqNfWZpaJNWj2roY3V5PbwdXwm0Gr5w6vooD2fP9mPLodPxqMLii5Scn45RLB",
    "2mdL6KTf7hkxeJ/jZg3cobWtDTOXW0/gufnmlXIPt8SWyHvPlsJXk2GlnLzFbTz3E9dXr7MvecyO6u3127M3vMxH6rV10Lzcbjmc",
    "GT5TI3IrYiwWdcwbU8VyiMr1dccR2urFu+lNKN3S6q1n8XJzZRsn1OomTm5BC0fvRZ96Ewk3z6FSUFYHuM0/cUKpNNMWjoktvBab",
    "xHyWGRitUShAgOoagXxp/FhiWtm8iragS+LzbK4voC3d/M3k7PagcMhcndfYhFOpsmq1nf/3Tv73Tyz/6zbtg+7hYbv7cCf/+wIe",
    "JRf5fev40P2fzf2C/O8Q8d9alLqT/32O594f9hZJvDfywz0ZXon5Kp1GYQcCssegyk/UnSaPX75g/hVyEDY4UkI4P0TQGDccE/G3",
    "glVW+UoHPlo178lwjRSu0WCqjIiHNJ0nvb29FeVc2CO59+KJBWbWvW4oCPvvKf3XvQnNiXsh9+R8H5QySrJU8q/qos49olYpvYNg",
    "939Z+EQk9YzXHkE7PX77vC6OX799fvbm9MVj5/j0hfP9yZ8FEWd1MUaAKxZtxaqouhQpvU4/WXjoxhfs6PchEWJBEm9rxij3SYaQ",
    "TFMiM9cP1eXneSBqNpcyFdnH8cUC7T1l/8Iq4vbrW0b61jku8phGHKwaLvtq5hQLpirXLJg7t90xsXMaVtXSw27VzZCMtcBoKoN5",
    "HxeKvF2MpPjh7CUETfCPCPQyQZ/vhhoxTOqZuwjSvvXmh7eWgan9F0GbgtEvC3rvhgiJWBGkflcwaSzge50BZes7dttK5XtgQr5S",
    "hKlkjvDd9D1E/DcJWnw0NNVlwm4GcDd4s+IJRLqayz577JnaDu4ux7XlOVmcaUZvQ1QPP7tr+uOmatuKRrs5flS7u1mKC7wDPksy",
    "00gULsWt6ny9zQtzs1riC0gOqDIV7Brv1VqlJDwyUuHcLI6p+T7y2rwKczJeMy99JrU5QznmXSYeLeRgn8p66bYHBRu/CulGtKU/",
    "mtdCDoxsAbCSPYCnzl+VsLcsZy7IghVsLbwoXAph5CQ/hU+oqL1FPgaB143p8e1eTw9w+eKEbc57f//b/xE3nhYewO4G98M6eCVM",
    "BhGo4wDJOI6WgiqMsyMH/x2fHf+34/+K/F/3aN8+aHYOW/uHuw3/Jdh/rJ/Tn5//67b3W4b/OzjoHMD+Ayzhjv/7HPyfeBTrcNhQ",
    "YasY2FIgGEOQmVwg+ne8IGoPtMU0Wmr6I8DlZXydCDR4duWeeMyhGjK10N7c9aDZfH5ydqLuXANZQjRWbpzBN4eqCzT9X5lzslQo",
    "sSUH4LxA6BTiDomNNEBBk8CetCesVzBxlTER9FbRndy49eRxtiLlsqPZqpnrTX2+YEbZDBOkR2wivOaW/rDZzAaEbY6fR/Es+tVv",
    "qEsMcZkMjQCBgTFrT2wPBnWPfZ0WIaK+VDk0iA53DKgw9W0sEVQ345xZxQVKrmQOS02ENexTfiwCupz6qRRVwKZRxLdHj549OztD",
    "yXvi+3VTdWVDrlyE2s32gfKln8MtyYfdBa7ie3R2/PqJzqhmK5RRaDPAF0GwgI8++xGcxhHfN+PORuyV8ZTvDFPm7rY4R6wwLBSa",
    "tljdNziV1wCz7j6gqMuG6lqz+7j79NBai/3ENYjqvaeHj7vNbq1YoN15+qjZXitwEeOORirQbD991O7UuMPoXELcBtb0qsIsg7qp",
    "rrNtutIpItHpPCp8qxdHQdAI5IU/8nEXLKac7Yx77CpvbMB7mQl4KQqYipJCQKhpdR4EvkkzdpOUCmdOCAi5veBFWXQgWmvkvcxZ",
    "DjeWuUEi/trRZhPE1KloRhXFuchZ9Iuv4bBSs2lAKN4vXsBIm8Nr18VELvfCSKgylQpr/TA9Ra1fD+xiqS0vTjVvyVdBKvMD4wtB",
    "zWHuUlRVdWEUNkJ5EaU+nNNqGWy+zdZhAE4qeWde6KGoaPU1GqLD7LAKj8YZcQmhGM0iHvQ4Dsd6ADaECtSOb3nohGqzr6L7reqi",
    "1Z/E0a8yrNko32jaD7GdaB/OsGBjqSbPiUInoZ+SZiQ1jcvYq8I1oLQejvTwZymHkGG99wpPmmPKsv3qA9TFXeEI1RJXS8KhsJGF",
    "Tk/5VmCBCwqy+woqhIqfuj7HlZjnMf3Y/mURIPBe8eJTbDF2FlrGEQxKnrKMQF85mJkEQGtOGBVYPpAT9sCDXd07wuMI/4MbGbkx",
    "SmZTFzGjHSiSYxpGwguPaY03CCUj5BxHxOIYePB7ou3JStH4StIpoqx2lCUPXx/qigQCJJr148SdTy1OSxJ3Uk6aMBZiMQylK5z0",
    "Fm/46Bfx11zhL2Tbhte4QHIZRQFynPMPJF2wVyWhZY9vdnzGr6f8yvVHfNEbrdhQtQAdp4SnnIApMbfV6uguQC3KsqahjXnWrqBQ",
    "01S6Vnr9Vmq+mhNmQBU3uwr3O4SG3G909sVKurGI1O27bJWkLrxW0bf5zkvMw9iPOaoHYh1i0mdy3JBXOMgDPpGAIRAIE5hzkuIa",
    "AF0yWNGiXUHah8WBYxa3h5xyHW6gjpDpak6n/TlfIKqFwmHhwm0d3YyvSaVllJi7Wdf7ae+Ykd2ze3bP7tk9u2f37J7ds3t2z+7Z",
    "Pbtn9+ye3bN7ds/u2T27Z/fsnt2ze3bP7tk9u+fu5/8BlPcKkwDwAAA=",
]
blob = base64.b64decode("".join(BLOB_PARTS))
tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz").extractall(".")
print("\u2705 Engine loaded.")

### 3️⃣ Your settings — edit the two lines, then click ▶

- **VIDEO_URL** — paste a YouTube link, e.g. a Flock Talk episode
- **ANTHROPIC_API_KEY** — get one at [console.anthropic.com](https://console.anthropic.com) → API Keys → Create Key (starts with `sk-ant-`)

In [ ]:
VIDEO_URL = "https://youtu.be/PASTE_YOUR_VIDEO_LINK"  #@param {type:"string"}
ANTHROPIC_API_KEY = "sk-ant-PASTE_YOUR_KEY"           #@param {type:"string"}
MAX_CLIPS = 3                                          #@param {type:"integer"}
print("✅ Saved. Video:", VIDEO_URL, "| clips:", MAX_CLIPS, "— go to step 4.")

### 4️⃣ Make the clips. Click ▶ and let it cook (5–15 min; the first run also downloads the speech model).

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["CLIP_ENGINE_ASR"] = "faster_whisper"

from pathlib import Path
from clip_engine.render import process

clips = process(source=VIDEO_URL, out_dir=Path("OUT"), work_root=Path("work"),
                mode="talk", max_clips=int(MAX_CLIPS))
print(f"\n✅ Done — {len(clips)} clips made. Go to step 5 to download.")
for c in clips: print("   •", c.name)

### 5️⃣ Download your clips. Click ▶ — `clips.zip` saves to your computer.

In [ ]:
!zip -qr clips.zip OUT
from google.colab import files
files.download("clips.zip")
print("✅ If no download started: click the 📁 folder icon on the left, right-click clips.zip → Download.")

---
**Tweak the look:** open the 📁 folder icon on the left → `config` → double-click `brand.yaml` — fonts, highlight colors, clip length all live there. Re-run step 4 after editing.

**Something errored?** Copy the red text and paste it to Claude — it built this and will fix it.